# Forecast ensemble time series and initial shock

Regional time series of every hindcast member from the initial state (lead 0) through the forecast, with diagnostics of the adjustment right after initialization: the regional mean absolute tendency |dX/dt| against its settled level and against ERA5, the ensemble-mean bias drift, RMSE and spread growth, and the spatial extremes of the members. A summary table condenses each experiment's shock into a few numbers.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states. For pairwise experiment-minus-baseline decay fits see `07_compute_and_plot_decay_timescales.ipynb`.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")  # observations (shared archive)
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")  # land mask
workflow_name = "fcst_atm_ensemble_ts"

run_segment = "fc"
# Forecast cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"),  # 201201-201202
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # 201206-201207
}
variables = ("PSL", "TS", "TREFHT", "PRECT")
frequency = "6hourly"  # "daily" also works; 6-hourly resolves the first-day adjustment
regions = ("global", "land", "ocean", "Tropics")
use_reference = True  # ERA5 6-hourly for PSL, TS, TREFHT, PRECT; other variables are model-only

# Shock diagnostics (lead days)
shock_days = 2.0  # window treated as the initial shock
settled_days = (10.0, 30.0)  # window taken as the adjusted state
tolerance = 0.2  # adjustment lasts while the 1-day running |dX/dt| exceeds (1 + tolerance) x settled
zoom_days = 10  # x range of the shock-focused figures

show_members = True
max_members = None  # e.g. 2 for a quick test; caches record the ensemble size
# Caches are rebuilt when missing, built with another ensemble size, or missing
# a requested region or the ERA5 reference; force_compute rebuilds them regardless.
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from IPython.display import display
import pandas as pd
from util.bias_metrics import experiment_registry
from util.forecast_ensemble_ts import FREQUENCIES, REGIONS, SUMMARY_METRICS, cache_ensemble_ts, ensemble_ts_cache_path, plot_ensemble_timeseries, plot_shock_diagnostics, plot_shock_summary, shock_summary
from util.model_evaluation import experiment_colors

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
if frequency not in FREQUENCIES:
    raise ValueError(f"frequency must be one of {FREQUENCIES}")
unknown = set(regions) - set(REGIONS)
if unknown:
    raise ValueError(f"Unknown regions {sorted(unknown)}; choose from {list(REGIONS)}")
landmask = LAND_SUPPORT_DIR / "landmask_1x1.nc"
reference_root = OBSERVATION_ROOT if use_reference else None
colors = experiment_colors(e for experiments in cases.values() for e in experiments)

## 3. Compute or reuse ensemble time series

In [ ]:
registries, results, failures = {}, {}, []
for season, experiments in cases.items():
    registry = registries[season] = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
    unknown = set(experiments) - set(registry)
    if unknown:
        raise ValueError(f"{season}: {sorted(unknown)} unavailable; choose from {list(registry)}")
    for variable in variables:
        for experiment in experiments:
            meta = registry[experiment]
            target = ensemble_ts_cache_path(data_dir, experiment, variable, frequency, meta["period"])
            try:
                ds = cache_ensemble_ts(
                    target, config=meta, experiment=experiment, variable=variable,
                    period=meta["period"], frequency=frequency, regions=regions,
                    landmask_file=landmask, reference_root=reference_root,
                    max_members=max_members, force_compute=force_compute,
                )
            except Exception as error:  # keep going; report at the end
                failures.append((season, variable, experiment, f"{type(error).__name__}: {error}"))
                print(f"[FAIL] {season} {variable} {experiment}: {type(error).__name__}: {error}")
                continue
            results.setdefault(season, {}).setdefault(variable, {})[experiment] = ds
if failures:
    print(f"\n{len(failures)} case(s) failed and are skipped below:")
    for season, variable, experiment, error in failures:
        print(f"  {season} {variable} {experiment}: {error}")

## 4. Ensemble time series

In [ ]:
# Thin lines are members, thick lines ensemble means, dashed black ERA5.
for season, by_variable in results.items():
    registry = registries[season]
    for variable, datasets in by_variable.items():
        labels = {name: registry[name]["label"] for name in datasets}
        for region in regions:
            fig = plot_ensemble_timeseries(
                datasets, region, labels=labels, colors=colors, zoom_days=zoom_days,
                show_members=show_members,
                title=f"{variable} {region} ({season} {run_segment}, {frequency})",
            )
            output = figure_dir / f"{variable}_{region}_{season}_{run_segment}_{frequency}_ensemble_ts.png"
            fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
            print(f"[FIGURE] {output}")
            if not show:
                plt.close(fig)

## 5. Initial-shock diagnostics

In [ ]:
# Tendency (ERA5 dashed), bias drift, RMSE (solid) and spread (dotted), spatial extremes.
for season, by_variable in results.items():
    registry = registries[season]
    for variable, datasets in by_variable.items():
        labels = {name: registry[name]["label"] for name in datasets}
        for region in regions:
            fig = plot_shock_diagnostics(
                datasets, region, labels=labels, colors=colors, zoom_days=zoom_days,
                title=f"{variable} {region}: initial shock ({season} {run_segment}, {frequency})",
            )
            output = figure_dir / f"{variable}_{region}_{season}_{run_segment}_{frequency}_shock_diagnostics.png"
            fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
            print(f"[FIGURE] {output}")
            if not show:
                plt.close(fig)

## 6. Shock summary

In [ ]:
tables = []
for season, by_variable in results.items():
    for variable, datasets in by_variable.items():
        table = shock_summary(datasets, shock_days=shock_days, settled_days=settled_days,
                              tolerance=tolerance)
        tables.append(table.assign(season=season))
summary = pd.concat(tables, ignore_index=True)
table_path = data_dir / f"shock_summary_{run_segment}_{frequency}.csv"
summary.to_csv(table_path, index=False)
print(f"[TABLE] {table_path}")
display(summary.round(3))

for (season, region), table in summary.groupby(["season", "region"], sort=False):
    fig = plot_shock_summary(
        table, list(SUMMARY_METRICS), colors=colors,
        title=f"Initial-shock summary: {region} ({season} {run_segment}, {frequency})",
    )
    output = figure_dir / f"shock_summary_{region}_{season}_{run_segment}_{frequency}.png"
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {output}")
    if not show:
        plt.close(fig)

## Notes: lead time, diagnostics, and summary metrics

**Lead time.** Each hindcast is one continuous run from the analysis time. The first model record has zero width and is the initial state (lead 0). Later records are placed at the midpoint of their output interval for interval means (`time: mean`, e.g. TS, TREFHT, PRECT) and at the interval end for instantaneous values (`time: point`, e.g. PSL, PS). For flux and process variables (e.g. PRECT) the lead-0 value is the first model step, not a balanced state, so their `initial_bias` and first tendency mostly show model spin-up.

**Reference.** ERA5 6-hourly samples on the same 180x360 grid, averaged with trapezoid weights over each model interval (the sample at that time for lead 0 and instantaneous variables). For 6-hourly model means this is the mean of the two bounding samples, an approximation. Only `PSL`, `TS`, `TREFHT`, and `PRECT` have a reference; other variables in `ts/<frequency>/` (e.g. `PS`, `TMQ`, `FLUT`) give model-only diagnostics.

| Diagnostic | Definition |
|---|---|
| `member_mean`, `member_min`, `member_max` | cos(lat)-weighted regional mean, and regional min/max, per member |
| `member_tendency` | regional mean of the gridpoint \|X(t) - X(t-1)\| / dt, per member (units per day) |
| `ensemble_spread` | regional mean of the gridpoint ensemble standard deviation |
| `reference_mean`, `reference_tendency` | the same for ERA5 |
| `ensemble_mean_rmse` | gridpoint RMSE of the ensemble mean against ERA5 over the region |

| Summary metric | Definition |
|---|---|
| `tendency_ratio` | ensemble-mean \|dX/dt\| over leads (0, `shock_days`] divided by its mean over `settled_days`; > 1 means the model adjusts faster right after initialization than once settled |
| `tendency_ratio_era5` | the same shock-window tendency divided by ERA5's over the same dates; separates the shock from the observed weather |
| `adjustment_days` | last lead before `settled_days` where the 1-day running \|dX/dt\| exceeds (1 + `tolerance`) x the settled level; 0 = no detectable shock |
| `initial_bias` | ensemble-mean minus ERA5 regional mean at lead 0 (analysis error) |
| `drift` | bias change from lead 0 to the last day of the shock window |
| `spread_ratio` | ensemble spread on the last day of the shock window divided by the spread at lead 0 |

The 6-hourly tendency includes the diurnal cycle and atmospheric tides; both the settled level and ERA5 contain them too, so the ratios compare like with like. Absolute tendencies of instantaneous and interval-mean variables are not comparable with each other.

Caches: `<experiment>_<variable>_<frequency>_<period>_ensemble_ts.nc` (dims member, lead, region); table: `shock_summary_<run_segment>_<frequency>.csv`.